# Mao et al. (2010): RMs towards the north Galactic pole

Rotation measures of extragalactic sources from [Mao et al. 2010, ApJ 714, 1170](https://doi.org/10.1088/0004-637X/714/2/1170), Table 1. The table is bundled with the package. This notebook runs in the test suite and in the documentation build.

In [ ]:
from __future__ import annotations

from importlib import resources

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table

from structurefunction import StencilOptions, structure_function
from structurefunction.fitting import fit_sf
from structurefunction.plotting import plot_sf

with resources.as_file(
    resources.files("structurefunction.data") / "mao2010_ngp.ecsv"
) as path:
    table = Table.read(path)

# Sources Mao et al. kept for their Section 5.2 analysis.
table = table[table["used_s5_2"]]
coords = SkyCoord(table["ra"], table["dec"])
rm = table["rm"].quantity
rm_err = table["rm_err"].quantity
len(table)

## Two-point structure function

The noise bias $\sigma_1^2 + \sigma_2^2$ is subtracted pair by pair. Errors come from a jackknife over 32 sky patches.

In [ ]:
bins = np.geomspace(0.5, 20, 9) * u.deg
sf2 = structure_function(rm, rm_err, coords, bins=bins, patches=32, rng=1)
sf2.to_table()["separation", "sf", "sf_err", "count"]

In [ ]:
fig, ax = plt.subplots()
plot_sf(sf2, ax=ax);

At separations beyond the correlation length the SF flattens at $2(\mathrm{Var(RM)} - \langle\sigma^2\rangle)$, twice the intrinsic variance (dashed line).

In [ ]:
pull = (sf2.sf - sf2.saturation) / sf2.sf_err
print(np.round(pull, 2))
assert np.all(np.abs(pull[-3:]) < 3)

## Three-point structure function

$\langle[\mathrm{RM}(x_+) - 2\mathrm{RM}(x_0) + \mathrm{RM}(x_-)]^2\rangle$ (Seta et al. 2023). Triplets are accepted when the outer sources are within 15° of collinear with the middle source and their separations agree to 20%. The plateau is three times that of the two-point SF.

In [ ]:
stencil = StencilOptions(angle_tol_deg=15.0, sep_tol=0.2, mode="exact")
sf3 = structure_function(
    rm, rm_err, coords, n_point=3, bins=bins, patches=32, stencil=stencil, rng=1
)
fig, ax = plt.subplots()
plot_sf(sf3, ax=ax);

## Fitting

A broken power law fitted with the full jackknife covariance, using the Sellentin & Heavens (2016) likelihood to account for noise in the covariance estimate.

In [ ]:
fit = fit_sf(
    sf2, model="broken_power_law", sampler="emcee", n_steps=3000, n_burn=1000, rng=2
)
for name, lo, med, hi in zip(fit.names, fit.lower, fit.median, fit.upper, strict=True):
    print(f"{name}: {med:.2f} (+{hi - med:.2f} / -{med - lo:.2f})")

In [ ]:
fig, ax = plt.subplots()
plot_sf(sf2, fit=fit, ax=ax, rng=3);